In [1]:
# Import required libraries
import sys
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

# Import the LLM debiasing analyzer
from LLM_debias2 import LLMPositionBiasAnalyzer

print("📚 Libraries imported successfully!")
print(f"📅 Experiment started at: {datetime.now()}")


📚 Libraries imported successfully!
📅 Experiment started at: 2025-08-09 22:26:00.571631


In [2]:
# metadata_json_file = 'data/music/meta_CDs_and_Vinyl.jsonl'
# with open(metadata_json_file, 'r', encoding='utf-8') as f:
#     for i, line in enumerate(f):
#         if i >= 10:  # change this to whatever number you want
#             break
#         data = json.loads(line)
#         print(data)

In [3]:
# Step 1: Read the ratings CSV (correct column order)
ratings_data_file = 'data/music/ratings_CDs_and_Vinyl.csv'
ratings_df = pd.read_csv(ratings_data_file, header=None, names=['UserID', 'item', 'rating', 'Timestamp'])

# Step 2: Filter ratings >= 3 (only keep items user liked)
print(f"📊 Original ratings: {len(ratings_df)}")
print(f"📊 Rating distribution:")
print(ratings_df['rating'].value_counts().sort_index())

# Only keep ratings >= 3
ratings_df = ratings_df[ratings_df['rating'] >= 3]
print(f"📊 After filtering ratings >= 3: {len(ratings_df)}")

# Step 3: Read the JSONL metadata
metadata_json_file = 'data/music/meta_CDs_and_Vinyl.jsonl'
item_title_map = {}

with open(metadata_json_file, 'r') as f:
    for line in f:
        try:
            data = json.loads(line)

            if 'parent_asin' not in data or 'title' not in data:
                continue

            asin = data['parent_asin'].strip()
            title = data['title'].strip()

            # Categories (remove "CDs & Vinyl" from nested lists)
            categories = data.get('categories', [])
            filtered_categories = [cat for cat in categories if cat != 'CDs & Vinyl']
            categories_str = ','.join(filtered_categories) if filtered_categories else 'N/A'

            # Price
            price = data.get('price', 'N/A')
            # Product Details
            details = data.get('details', {})
            release_date = details.get('Original Release Date', 'N/A')
            country_origin = details.get('Country of Origin', 'N/A')
            run_time = details.get('Run time', 'N/A')

            # Title Enhancement
            full_title = (
                f"{title}; "
                f"CATEGORIES:{categories_str}; "
                f"PRICE:{price}; "
            )

            item_title_map[asin] = full_title

        except json.JSONDecodeError:
            continue


# Step 4: Map ASIN -> Title
ratings_df['Title'] = ratings_df['item'].astype(str).map(item_title_map)

# Step 5: Filter only those rows that matched
filtered_df = ratings_df.dropna(subset=['Title'])

# Step 6: Sort by UserID and Timestamp to ensure proper ordering
filtered_df = filtered_df.sort_values(['UserID', 'Timestamp'])

# Step 7: Final dataset - now only contains items with ratings >= 3
final_df = filtered_df[['UserID', 'Title', 'Timestamp']]

print(f"📊 Final dataset: {len(final_df)} interactions")
print(f"📊 Unique users: {final_df['UserID'].nunique()}")
print(f"📊 Unique items: {final_df['Title'].nunique()}")
print("\n🎯 Sample of processed data (ratings >= 3 only):")
print(final_df.head())

# Save the processed data
output_path = 'data/music/user_title_timestamp_metadata_4.csv'
final_df.to_csv(output_path, index=False)
print(f"\n💾 Saved processed data to {output_path}")

# Show user interaction distribution
user_interactions = final_df.groupby('UserID').size()
print(f"\n📊 User interaction distribution (after filtering):")
print(f"  Mean interactions per user: {user_interactions.mean():.2f}")
print(f"  Median interactions per user: {user_interactions.median():.2f}")
print(f"  Users with >= 6 interactions: {sum(user_interactions >= 6)}")
print(f"  Users with >= 10 interactions: {sum(user_interactions >= 10)}")

📊 Original ratings: 3749004
📊 Rating distribution:
rating
1.0     160131
2.0     132321
3.0     264825
4.0     672525
5.0    2519202
Name: count, dtype: int64
📊 After filtering ratings >= 3: 3456552
📊 Final dataset: 3063826 interactions
📊 Unique users: 1359064
📊 Unique items: 355891

🎯 Sample of processed data (ratings >= 3 only):
                        UserID  \
483931    A0001624UKLQG4OFIM8X   
2153907  A0002382258OFJJ2UYNTR   
3481450  A0002382258OFJJ2UYNTR   
605131    A0005916MHK9RK69491E   
3543000   A0006650PUSTDDZX7UKW   

                                                     Title   Timestamp  
483931   The Band Last Waltz; CATEGORIES:Classic Rock,A...  1361059200  
2153907  Hymns: My Life; CATEGORIES:Christian & Gospel,...  1366156800  
3481450  Eye On It; CATEGORIES:Christian & Gospel,Gospe...  1366156800  
605131   Empire; CATEGORIES:Rock,Progressive,Progressiv...  1404432000  
3543000  Lindsey Stirling; CATEGORIES:Dance & Electroni...  1379548800  

💾 Saved processed data 

In [2]:
final_df = pd.read_csv('data/music/user_title_timestamp_metadata_4.csv')
print(final_df.head())

                  UserID                                              Title  \
0   A0001624UKLQG4OFIM8X  The Band Last Waltz; CATEGORIES:Classic Rock,A...   
1  A0002382258OFJJ2UYNTR  Hymns: My Life; CATEGORIES:Christian & Gospel,...   
2  A0002382258OFJJ2UYNTR  Eye On It; CATEGORIES:Christian & Gospel,Gospe...   
3   A0005916MHK9RK69491E  Empire; CATEGORIES:Rock,Progressive,Progressiv...   
4   A0006650PUSTDDZX7UKW  Lindsey Stirling; CATEGORIES:Dance & Electroni...   

    Timestamp  
0  1361059200  
1  1366156800  
2  1366156800  
3  1404432000  
4  1379548800  


In [5]:
# The final_df is already created above with ratings >= 3 filtering
# Let's verify the data structure
print("🔍 Verifying processed data structure:")
print(f"📊 Shape: {final_df.shape}")
print(f"📊 Columns: {list(final_df.columns)}")
print(f"📊 Data types:")
print(final_df.dtypes)
print(f"\n📊 Sample user history (first user):")
sample_user = final_df['UserID'].iloc[0]
user_history = final_df[final_df['UserID'] == sample_user].sort_values('Timestamp')
print(f"User {sample_user} has {len(user_history)} interactions (all rated >= 3)")
print(user_history[['Title', 'Timestamp']].head(10))

🔍 Verifying processed data structure:
📊 Shape: (3063826, 3)
📊 Columns: ['UserID', 'Title', 'Timestamp']
📊 Data types:
UserID       object
Title        object
Timestamp     int64
dtype: object

📊 Sample user history (first user):
User A0001624UKLQG4OFIM8X has 1 interactions (all rated >= 3)
                                               Title   Timestamp
0  The Band Last Waltz; CATEGORIES:Classic Rock,A...  1361059200


In [3]:
analyzer = LLMPositionBiasAnalyzer(
    data=final_df,
    data_name="music",
    model="gpt-3.5-turbo",
    backend="openai",
    list_size = 20,
    api_tier="tier_2"
)

📊 User filtering results:
  Total users in dataset: 1359064
  Users with ≥6 items: 68163
  Filtered out: 1290901 users
✅ Selected 5 bias users and 200 evaluation users
   All selected users have ≥6 items for reliable evaluation
Initialized LLM Bias Analyzer:
  Model: gpt-3.5-turbo
  Backend: openai
  API Tier: tier_2
  Rate Limits: 5000 RPM, 2000000 TPM
  Max Workers: 25
  Batch Size: 50
  Request Delay: 0.030s


In [7]:
bias_analysis =  analyzer.compute_bias_analysis(5,None,True,None,20)
print(bias_analysis)

Bias users: ['AEQMXKCI6NTOQ', 'A25DJYKHTCOX9S', 'A135BINUTM0ZNW', 'A8FOD1I51N1T7', 'A3HYV4QF8YCZH5']
\nCalculating bias scores...


Bias detection:   0%|                                     | 0/5 [00:00<?, ?it/s]


Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50



Bias detection:  20%|█████▊                       | 1/5 [00:07<00:28,  7.02s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.780
  Average recency items in top 10%: 0.540
  Average middle items in top 10%: 0.680

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50


...

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.720
  Average recency items in top 10%: 0.380
  Average middle items in top 10%: 0.900

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50



Bias detection:  80%|███████████████████████▏     | 4/5 [00:23<00:05,  5.53s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.380
  Average recency items in top 10%: 0.620
  Average middle items in top 10%: 1.000

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50



Bias detection: 100%|█████████████████████████████| 5/5 [00:29<00:00,  5.88s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.960
  Average recency items in top 10%: 0.500
  Average middle items in top 10%: 0.540
{'bias_scores': {'avg_primacy': np.float64(0.6599999999999999), 'avg_recency': np.float64(0.48), 'avg_middle': np.float64(0.86)}, 'propensity_scores': {1: 0.7261490370736909, 2: 0.7609937044363775, 3: 0.7950399763409617, 4: 0.8280364855102174, 5: 0.8597309959501938, 6: 0.8898735542044357, 7: 0.918219730135688, 8: 0.9445338893900183, 9: 0.968592436883306, 10: 0.9901869692616228, 11: 1.0091272744120166, 12: 1.0252441177807488, 13: 1.0383917584868099, 14: 1.0484501429456454, 15: 1.0553267298396851, 16: 1.0589579076413147, 17: 1.0593099743220558, 18: 1.0563796581453275, 19: 1.0501941682875764, 20: 1.0408107741923882}, 'avg_bias_result': {'avg_primacy': np.float64(0.6599999999999999), 'avg_recency': np.float64(0.48), 'avg_middle': np.float64(0.86)}, 'experiment_results': {'avg_primacy': np.fl

In [4]:
# prebias_gpt35_music = {'avg_primacy': 0.9119999999999999,
#  'avg_recency': 0.136,
#  'avg_middle': 0.952}

# prebias_gpt35_music = {'avg_primacy': 0.34,
#  'avg_recency': 0.42400000000000004,
#  'avg_middle': 1.236}

prebias_gpt35_music = {'avg_primacy': 0.384,
 'avg_recency': 0.716,
 'avg_middle': 0.9}

prebias_gpt35_music_meta = {'avg_primacy': 0.5439999999999999,
 'avg_recency': 0.544,
 'avg_middle': 0.9120000000000001}

prebias_gpt35_music_meta2 = {'avg_primacy': 0.6599999999999999,
 'avg_recency': 0.48,
 'avg_middle': 0.86}

In [ ]:
# Step 1: Configure experiment parameters and run the complete evaluation
print("\n🔧 COMPLETE DEBIASING EXPERIMENT")
print("=" * 50)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 20        # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Run the complete evaluation pipeline
# This will automatically:
# 1. Split users into bias detection and evaluation sets
# 2. Run bias detection on bias detection users
# 3. Calculate propensity scores from detected bias
# 4. Evaluate on evaluation users using the calculated propensity scores
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    batch_size=batch_size,
    use_parallel=True,
    precalculated_bias=prebias_gpt35_music_meta2,
    checkpoint_file = "evaluation_checkpoint_bias20_music_meta2_4.json"
)

print("\n✅ EVALUATION COMPLETED!")



🔧 COMPLETE DEBIASING EXPERIMENT
🎯 Candidates per evaluation: 20
🔄 Trials per user: 20
📦 Batch size: 20

🚀 RUNNING COMPLETE EVALUATION PIPELINE...
This will:
1. 📊 Select separate users for bias detection vs evaluation
2. 🔍 Run bias detection on bias detection users
3. ⚖️ Calculate propensity scores from detected bias
4. 📈 Evaluate on evaluation users using calculated propensity scores
5. 💾 Save all raw data for future reanalysis
📁 Checkpoint file: evaluation_checkpoint_bias20_music_meta2_4.json
📂 Resuming from checkpoint: 0 users already completed
API Tier: tier_2 (RPM: 5000, TPM: 2000000)
Max workers - Bias: 25, Trials: 12, Users: 3
📊 Using bias analysis from checkpoint
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/10 (20 users)
Evaluating 20 users in parallel with max_workers=3...

Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50

Runn

Trials (batch 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]

Trials (batch 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]

Trials (batch 1/1):   5%|█▏                      | 1/20 [00:01<00:19,  1.04s/it]

Trials (batch 1/1):  50%|███████████▌           | 10/20 [00:01<00:00, 10.95it/s]

Trials (batch 1/1):  45%|██████████▊             | 9/20 [00:01<00:01,  8.80it/s]

Trials (batch 1/1):  70%|████████████████       | 14/20 [00:02<00:01,  5.99it/s]

Trials (batch 1/1):  85%|███████████████████▌   | 17/20 [00:02<00:00,  8.04it/s]

Trials (batch 1/1): 100%|███████████████████████| 20/20 [00:03<00:00,  6.59it/s]


Completed 20 successful trials out of 20 attempted
Completed 20 successful trials out of 20 attempted



Trials (batch 1/1): 100%|███████████████████████| 20/20 [00:03<00:00,  5.79it/s]


Completed 20 successful trials out of 20 attempted

Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50


...


Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50



Trials (batch 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]


Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50




Trials (batch 1/1):   5%|█▏                      | 1/20 [00:00<00:16,  1.15it/s]

Trials (batch 1/1):  20%|████▊                   | 4/20 [00:01<00:03,  4.87it/s]

Trials (batch 1/1):  50%|███████████▌           | 10/20 [00:01<00:00, 11.01it/s]

Trials (batch 1/1):  55%|████████████▋          | 11/20 [00:01<00:01,  8.60it/s]

Trials (batch 1/1):  45%|██████████▊             | 9/20 [00:01<00:01,  6.72it/s]

Trials (batch 1/1):  60%|█████████████▊         | 12/20 [00:01<00:01,  7.47it/s]

Trials (batch 1/1):  80%|██████████████████▍    | 16/20 [00:02<00:00,  7.29it/s]

Trials (batch 1/1):  70%|████████████████       | 14/20 [00:02<00:00,  6.29it/s]

Trials (batch 1/1):  90%|████████████████████▋  | 18/20 [00:02<00:00,  7.70it/s]

Trials (batch 1/1): 100%|███████████████████████| 20/20 [00:03<00:00,  6.32it/s]

Completed 20 successful trials out of 20 attempted
Completed 20 successful trials out of 20 attempted


In [ ]:
bias_analysis_100 =  analyzer.compute_bias_analysis(5,None,True,None,100)


In [9]:
print(bias_analysis_100)

{'bias_scores': {'avg_primacy': 0.14800000000000002, 'avg_recency': 0.20400000000000001, 'avg_middle': 1.6480000000000001}, 'propensity_scores': {1: 2.562030222876484, 2: 2.631060511770599, 3: 2.7004725958257265, 4: 2.7701996005789074, 5: 2.840172382785748, 6: 2.9103196250762973, 7: 2.9805679390489606, 8: 3.050841976672992, 9: 3.1210645498346725, 10: 3.1911567578267084, 11: 3.2610381225448357, 12: 3.3306267311202737, 13: 3.399839385681719, 14: 3.4685917599061815, 15: 3.536798561984355, 16: 3.6043737035934904, 17: 3.671230474439194, 18: 3.7372817218972925, 19: 3.8024400352581025, 20: 3.866617934048306, 21: 3.929728059880282, 22: 3.991683371255402, 23: 4.05239734072653, 24: 4.111784153806013, 25: 4.169758908988879, 26: 4.226237818246857, 27: 4.281138407337446, 28: 4.33437971526352, 29: 4.385882492213061, 30: 4.435569395305569, 31: 4.483365181471584, 32: 4.5291968967946215, 33: 4.572994061650663, 34: 4.614688850989134, 35: 4.654216269111182, 36: 4.691514318315746, 37: 4.72652416080166, 38

In [16]:
prebias_gpt35_music_100 = {'avg_primacy': 0.14800000000000002,
 'avg_recency': 0.20400000000000001,
 'avg_middle': 1.6480000000000001}

In [17]:
# Step 1: Configure experiment parameters and run the complete evaluation
print("\n🔧 COMPLETE DEBIASING EXPERIMENT")
print("=" * 50)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 20        # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Run the complete evaluation pipeline
# This will automatically:
# 1. Split users into bias detection and evaluation sets
# 2. Run bias detection on bias detection users
# 3. Calculate propensity scores from detected bias
# 4. Evaluate on evaluation users using the calculated propensity scores
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    batch_size=batch_size,
    use_parallel=True,
    precalculated_bias=prebias_gpt35_music_100,
    checkpoint_file = "evaluation_checkpoint_bias20_music_4.json"
)

print("\n✅ EVALUATION COMPLETED!")


🔧 COMPLETE DEBIASING EXPERIMENT
🎯 Candidates per evaluation: 20
🔄 Trials per user: 20
📦 Batch size: 20

🚀 RUNNING COMPLETE EVALUATION PIPELINE...
This will:
1. 📊 Select separate users for bias detection vs evaluation
2. 🔍 Run bias detection on bias detection users
3. ⚖️ Calculate propensity scores from detected bias
4. 📈 Evaluate on evaluation users using calculated propensity scores
5. 💾 Save all raw data for future reanalysis
📁 Checkpoint file: evaluation_checkpoint_bias20_music_4.json
API Tier: tier_1 (RPM: 3500, TPM: 1000000)
Max workers - Bias: 15, Trials: 7, Users: 3
🔍 Computing bias analysis...
Bias users: ['A1BQV9OABXHUTD', 'A1Q2GLZXL2CNJ8', 'AIMGGXJENOIQ1', 'A38CPYF83EPG8Y', 'A2YR1GTIPUANDE']
Using precalculated bias scores...
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/10 (20 users)
Evaluating 20 users in parallel with max_workers=3...


User evaluation:   0%|                                   | 0/20 [00:00<?, ?it/s]


Running 20 randomization trials (with raw data preservation)...
Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=7...
Rate limiting: 7 workers, 0.050s delay, batch size: 25

Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=7...
Rate limiting: 7 workers, 0.050s delay, batch size: 25

Executing 20 trials in parallel with max_workers=7...
Rate limiting: 7 workers, 0.050s delay, batch size: 25



als (batch 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]

 (batch 1/1):   0%|                                | 0/20 [00:00<?, ?it/s]


als (batch 1/1):   5%|█▏                      | 1/20 [00:00<00:18,  1.04it/s]


atch 1/1):   5%|█▏                      | 1/20 [00:00<00:18,  1.03it/s]

 (batch 1/1):   5%|█▏                      | 1/20 [00:01<00:19,  1.01s/it]

als (batch 1/1):  15%|███▌                    | 3/20 [00:01<00:05,  3.06it/s]


atch 1/1):  15%|███▌                    | 3/20 [00:01<00:06,  2.59it/s]

als (batch 1/1):  25%|██████                  | 5/20 [00:01<00:03,  4.50it/s]


atch 1/1):  20%|████▊                   | 4/20 [00:01<00:05,  2.88it/s]


als (batch 1/1):  35%|████████▍               | 7/20 [00:01<00:03,  4.10it/s]


atch 1/1):  35%|████████▍               | 7/20 [00:02<00:02,  4.57it/s]

 (batch 1/1):  40%|█████████▌              | 8/20 [00:02<00:03,  3.23it/s]


als (batch 1/1):  45%|██████████▊             | 9/20 [00:02<00:03,  3.53it/s]

Completed 20 successful trials out of 20 attempted


...

Completed 20 successful trials out of 20 attempted

Running 20 randomization trials (with raw data preservation)...
Executing 20 trials in parallel with max_workers=7...
Rate limiting: 7 workers, 0.050s delay, batch size: 25




als (batch 1/1):  35%|████████▍               | 7/20 [00:01<00:02,  4.62it/s]

als (batch 1/1):  40%|█████████▌              | 8/20 [00:02<00:04,  2.81it/s]

als (batch 1/1):  50%|███████████▌           | 10/20 [00:03<00:02,  3.74it/s]

als (batch 1/1):  65%|██████████████▉        | 13/20 [00:03<00:01,  5.74it/s]

 (batch 1/1):  30%|███████▏                | 6/20 [00:01<00:02,  5.12it/s]

 (batch 1/1):  40%|█████████▌              | 8/20 [00:02<00:03,  3.68it/s]

 (batch 1/1):  45%|██████████▊             | 9/20 [00:02<00:02,  3.91it/s]

 (batch 1/1):  50%|███████████▌           | 10/20 [00:02<00:02,  4.57it/s]

als (batch 1/1):  75%|█████████████████▎     | 15/20 [00:04<00:01,  2.84it/s]

User evaluation:  95%|████████████████████████▋ | 19/20 [00:49<00:02,  2.63s/it]

 (batch 1/1):  75%|█████████████████▎     | 15/20 [00:04<00:01,  3.80it/s]

Completed 20 successful trials out of 20 attempted




 (batch 1/1):  80%|██████████████████▍    | 16/20 [00:04<00:01,  3.89it/s]

 (batch 1/1):  85%|███████████████████▌   | 17/20 [00:05<00:01,  2.58it/s]

 (batch 1/1):  90%|████████████████████▋  | 18/20 [00:05<00:00,  3.15it/s]

User evaluation: 100%|██████████████████████████| 20/20 [00:51<00:00,  2.56s/it]


Completed 20 successful trials out of 20 attempted
✅ Batch 10 completed. Progress: 200/200 users

📊 Computing final metrics from 200 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.1700 ± 0.3756
  NDCG@1:      0.1700 ± 0.3756
  NDCG@5:      0.3461 ± 0.3873
  NDCG@10:     0.4165 ± 0.3449
  NDCG@20:     0.4892 ± 0.2673
  Number of evaluations: 200

Benchmark Results (Accuracy) - From Paper:
Method          Movie Dataset  
------------------------------
Raw Output      0.2740±0.0593
Bootstrapping   0.2537
STELLA          0.2976
Our Method      0.1700±0.3756

Accuracy Comparison (Movie Dataset):
----------------------------------------
Our Method vs Raw Output:    -0.1040
Our Method vs Bootstrapping: -0.0837
Our Method vs STELLA:        -0.1276

NDCG Analysis:
----------------------------------------
NDCG@1 = Accuracy: 0.1700
NDCG@5:  0.3461 (203.6% of NDCG@1)
NDCG@10: 0.4165 (245.0% of NDCG@1)
NDCG@20: 0.4892 (287.8% of NDCG@1)

Summary:


In [12]:
bias_analysis_100 =  analyzer.compute_bias_analysis(100,None,True,None,20)

Bias users: ['A9DMTMLFR9CO5' 'AHG1GTQZUYNJN' 'A2TFO7NREP2B2D' 'A2YAPAG1IPNK7K'
 'AEKGGV851HY3K' 'A2MRQG8RN5JI7R' 'A12R54MKO17TW0' 'A1C7NPVPFF4OO8'
 'A22X72C51HQ7AS' 'A1JE8B5PU9ISMC' 'A1WX42M589VAMQ' 'A1BA3VR9EKXI87'
 'A3IWBNNKB5Y1ZY' 'A2CNQ98AQMTF1T' 'A293MJV7OBALZ8' 'AWA4RHNCRLQNS'
 'A1X15L7VXGNN6Z' 'A1OYXEHASP0RIU' 'A33BNNNLHJRAJV' 'A258R8TKIF1ONL'
 'A3J2KYGO0RYJ63' 'A6P41US94O8NS' 'A1ITN92CWNHKA3' 'A129A0AHSRTAJK'
 'A1WRYC2OGD5TSK' 'A2KPZ10PHV91G6' 'A2IDXYAQ8A0Q7M' 'A28TAEPW233MTU'
 'A2SL37IK3BCDC8' 'A20BS282KMRQ60' 'AVO85H3PRWWC3' 'AEPQL8RZJREBJ'
 'A1K3F17YG0OM2Q' 'A1OFY4ATO7D13W' 'A2KH83L1F70QR8' 'A11AHD1AHB6CH2'
 'A14KOSAV1RXQ8O' 'A24IIZC446AMGL' 'A1P9JMP3NIDUUU' 'A1BPYX701H98LN'
 'A1BGUZXTNCU3JN' 'AOMRY3FX1U5RV' 'A3EJWHGO91FA0N' 'A14BTJRH9VNLJJ'
 'A1VD7LQQVAYTW3' 'A2UZ22WGICMJE6' 'A1RJJ56MBJMX87' 'A14ZENEO7QG6ZI'
 'A2ME8IU4TOJ6WP' 'A3600EP59PMKNK' 'A4QB3WJGF267C' 'A2QOC4EV8HGZ9D'
 'AEGLF3184W7ND' 'A1BTUGG9C3VZEM' 'A12TRC9JF61YQ6' 'A9MAUO67ABVOB'
 'A18EM13OMDSNBK' 'ACL9N6JIKFBAB'

Bias detection:   0%|                                    | 0/10 [00:00<?, ?it/s]


Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=15...
Rate limiting: 15 workers, 0.050s delay, batch size: 25



Shuffles (batch 1/2): 100%|█████████████████████| 25/25 [00:06<00:00,  4.09it/s]


Batch 1 complete, waiting 1.0s before next batch...


...

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.100
  Average recency items in top 10%: 0.140
  Average middle items in top 10%: 1.760

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=15...
Rate limiting: 15 workers, 0.050s delay, batch size: 25



Shuffles (batch 1/2): 100%|█████████████████████| 25/25 [00:06<00:00,  3.94it/s]


Batch 1 complete, waiting 1.0s before next batch...



Bias detection: 100%|███████████████████████████| 10/10 [02:28<00:00, 14.88s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.180
  Average recency items in top 10%: 0.220
  Average middle items in top 10%: 1.600


In [13]:
bias_analysis_100

{'bias_scores': {'avg_primacy': 0.144,
  'avg_recency': 0.23200000000000004,
  'avg_middle': 1.6239999999999999},
 'propensity_scores': {1: 2.038063311859906,
  2: 1.782973628903709,
  3: 1.5815309787839016,
  4: 1.422381258858143,
  5: 1.2970594268630151,
  6: 1.1992487161540206,
  7: 1.124253335547588,
  8: 1.0686233070551319,
  9: 1.029889508870065,
  10: 1.0063803792551218,
  11: 0.9971011626750926,
  12: 1.0016634318318955,
  13: 1.0202578624423222,
  14: 1.0536675767662464,
  15: 1.1033233692690319,
  16: 1.1714062921639319,
  17: 1.2610079447763867,
  18: 1.3763650239708938,
  19: 1.5231931139497055,
  20: 1.7091565445052326},
 'avg_bias_result': {'avg_primacy': 0.144,
  'avg_recency': 0.23200000000000004,
  'avg_middle': 1.6239999999999999},
 'num_bias_users': 100,
 'precalculated_bias_used': False}

In [14]:
prebias_gpt35_music_100 = {'avg_primacy': 0.144,
 'avg_recency': 0.23200000000000004,
 'avg_middle': 1.6239999999999999}

In [15]:
# Step 1: Configure experiment parameters and run the complete evaluation
print("\n🔧 COMPLETE DEBIASING EXPERIMENT")
print("=" * 50)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 20        # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Run the complete evaluation pipeline
# This will automatically:
# 1. Split users into bias detection and evaluation sets
# 2. Run bias detection on bias detection users
# 3. Calculate propensity scores from detected bias
# 4. Evaluate on evaluation users using the calculated propensity scores
print("\n🚀 RUNNING COMPLETE EVALUATION PIPELINE...")
print("This will:")
print("1. 📊 Select separate users for bias detection vs evaluation")
print("2. 🔍 Run bias detection on bias detection users")  
print("3. ⚖️ Calculate propensity scores from detected bias")
print("4. 📈 Evaluate on evaluation users using calculated propensity scores")
print("5. 💾 Save all raw data for future reanalysis")

results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    batch_size=batch_size,
    use_parallel=True,
    precalculated_bias=prebias_gpt35_music_100,
    checkpoint_file = "evaluation_checkpoint_bias20_music.json"
)

print("\n✅ EVALUATION COMPLETED!")


🔧 COMPLETE DEBIASING EXPERIMENT
🎯 Candidates per evaluation: 20
🔄 Trials per user: 20
📦 Batch size: 20

🚀 RUNNING COMPLETE EVALUATION PIPELINE...
This will:
1. 📊 Select separate users for bias detection vs evaluation
2. 🔍 Run bias detection on bias detection users
3. ⚖️ Calculate propensity scores from detected bias
4. 📈 Evaluate on evaluation users using calculated propensity scores
5. 💾 Save all raw data for future reanalysis
📁 Checkpoint file: evaluation_checkpoint_bias20_music.json
📂 Resuming from checkpoint: 200 users already completed
API Tier: tier_1 (RPM: 3500, TPM: 1000000)
Max workers - Bias: 15, Trials: 7, Users: 3
📊 Using bias analysis from checkpoint
👥 Total users: 200, Completed: 200, Remaining: 0

📊 Computing final metrics from 8 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.2500 ± 0.4330
  NDCG@1:      0.2500 ± 0.4330
  NDCG@5:      0.2984 ± 0.4239
  NDCG@10:     0.3400 ± 0.4086
  NDCG@20:     0.4617 ± 0.3149
  Numbe